# FiLM Projector (LearnableProjectorAlpha)
FiLM-обусловленная проекция: гамма-модуляция входа по оси условия.

In [1]:
import sys
sys.path.insert(0, '..')

import gensim.downloader as api
import torch
import os

from src.data import load_data
from src.models import build_model
from src.train import train
from src.evaluate import run_all_evaluations
from src.visualize import plot_curves, visualize_3d_projection

os.makedirs('../results/film_gamma', exist_ok=True)

In [2]:
from gensim.models import KeyedVectors

embeddings = KeyedVectors.load_word2vec_format(
    '../gensim-data/glove-wiki-gigaword-300/glove-wiki-gigaword-300.gz'
)
print(f'Vocab size: {len(embeddings.key_to_index)}')

Vocab size: 400000


In [3]:
train_loader, val_loader, gender_clean, royalty_clean = load_data(embeddings)


--------------------------------------------------
  GENDER PAIRS
--------------------------------------------------
  candidates: 100 | valid ones: 100 | discarded: 0

--------------------------------------------------
  ROYALTY PAIRS
--------------------------------------------------
  candidates: 106 | valid ones: 106 | discarded: 0

  analysis: GENDER
  cos mean=0.4609 std=0.1898 | threshold 0.3: 83/100

  analysis: STATUS
  cos mean=0.2943 std=0.1196 | threshold 0.3: 51/106

  Cosine between axes: 0.3172

  Train: 106 | Val: 28


In [4]:
model = build_model('film_gamma')  # единственное отличие от baseline
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model, history = train(
    projector=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    lambda_ortho=0.1,
    num_epochs=1500,
)

torch.save(model.state_dict(), '../results/film_gamma/model.pt')

Training:   0%|          | 0/1500 [00:00<?, ?it/s]


Best model: epoch 1500, val_loss=0.0548


In [5]:
plot_curves(history)

In [6]:
metrics = run_all_evaluations(model, embeddings)
print(metrics)

Analigies:
  king - man + woman → queen  |  cos=0.8518  dist=0.5107
  prince - boy + girl → princess  |  cos=0.3970  dist=0.6121
  husband - man + woman → wife  |  cos=0.9577  dist=1.0534
  uncle - man + woman → aunt  |  cos=0.9737  dist=0.4968
  father - man + woman → mother  |  cos=0.9885  dist=0.3745
  brother - man + woman → sister  |  cos=0.7653  dist=0.5744
  king - prince + princess → queen  |  cos=0.8741  dist=0.4434
  grandfather - father + mother → grandmother  |  cos=0.9685  dist=0.3685
  actor - man + woman → actress  |  cos=0.9844  dist=0.3788
  he - man + woman → she  |  cos=0.9957  dist=0.2731
  analogy_cos=0.8757 | analogy_dist=0.5086
{'analogy_cos': 0.8756710365783216, 'analogy_dist': 0.5085712671279907}


### Visualization 
FiLM — gender axis vs status axis

In [8]:
for axis_name, condition in [('gender', [1,0,0]), ('status', [0,1,0])]:
    visualize_3d_projection(
        gender_pairs=gender_clean, 
        royalty_pairs=royalty_clean,
        embeddings=embeddings, 
        projector=model,
        condition=condition,
        output_html=f'../results/film_gamma/film_gamma_{axis_name}_3d.html',
        title=f'Film {axis_name.capitalize()} (gamma beta) Projection - 3D Visualization'
    )

Saved: ../results/film_gamma/film_gamma_gender_3d.html


Saved: ../results/film_gamma/film_gamma_status_3d.html
